#3. Notebook 02_silver_transformação — Tratamento dos dados recebidos

#### Objetivo
Ler **somente** `bronze_datajud_raw` (camada imutável) e reconstruir a camada
Silver por completo: parse do payload JSON, deduplicação de capturas,
tabelas de processos e movimentos, dimensões derivadas, regras de marcação,
classificação de eventos, marcos processuais e QA de execução.

#### Desenho
- **Entrada única:** `bronze_datajud_raw` (raw_payload, tribunal_origem, ingestion_timestamp)
- **Saídas (10 tabelas):** silver_datajud_processos, silver_fato_movimento,
  silver_dim_orgao_julgador, silver_dim_assunto, silver_ponte_processo_assunto,
  silver_dim_processo, silver_regras_marcacao,
  silver_datajud_eventos_enriquecidos, silver_marcos_processo, silver_qa_execucao
- **Padrões:** CREATE OR REPLACE (idempotente) + COMMENT (governança);
  regravação de schema habilitada (overwriteSchema) para evolução controlada
- **Idempotência:** o notebook pode ser re-executado ponta a ponta sem resíduos

#### Papel da camada silver 
Se a Bronze é o arquivo bruto do Datajud, a Silver é a primeira versão
*usável*: tipos corretos, 1 linha por processo, 1 linha por movimento e
eventos já classificados pelas regras de marcação. A Gold não reprocessa
nada — consome o que aqui está garantido.

### BLOCO 1 — Parse do raw (JSON → estruturas tipadas)

A Bronze guarda o payload bruto da API de interoperabilidade do CNJ em
`raw_payload` (string). Aqui definimos o **schema espelhado** da resposta
pública: processo, classe, sistema, formato, órgão julgador, assuntos e o
array de movimentos (com complementos tabelados).

**Por que schema explícito:** `from_json` com StructType garante tipagem
determinística, rejeita silênciosamente campos fora do contrato e documenta
a estrutura da fonte dentro do próprio código — parte do requisito de
catálogo e governança da disciplina.

**Saída do bloco:** DataFrame `bronze` (243.831 registros), ainda sem
qualquer filtragem — o parse não decide, só tipa.

In [0]:
# Databricks notebook source
# ==============================================================================
# 02_SILVER_TRANSFORMACAO
# Lê SOMENTE de bronze_datajud_raw (imutável) e monta a camada silver.
# Escopo essencial: 10 tabelas — processos, movimentos, dims derivadas,
# regras de marcação, eventos enriquecidos, marcos e QA.
# Padrões: CREATE OR REPLACE (idempotente, atualiza o catalog) + COMMENT.
# ==============================================================================

from pyspark.sql import functions as F, types as T, Window

# ==============================================================================
# BLOCO 1 — PARSE DO RAW (JSON -> estruturas tipadas)
# O bronze guarda o payload bruto do Datajud em raw_payload.
# O schema espelha a estrutura pública da API de interoperabilidade do CNJ.
# ==============================================================================

schema_comp = T.StructType([
    T.StructField("codigo",    T.StringType()),
    T.StructField("nome",      T.StringType()),
    T.StructField("descricao", T.StringType()),
])

schema_mov = T.StructType([
    T.StructField("codigo",                 T.IntegerType()),
    T.StructField("nome",                   T.StringType()),
    T.StructField("dataHora",               T.StringType()),
    T.StructField("complementosTabelados",  T.ArrayType(schema_comp)),
])

schema_proc = T.StructType([
    T.StructField("id",                   T.StringType()),
    T.StructField("numeroProcesso",       T.StringType()),
    T.StructField("dataAjuizamento",      T.StringType()),
    T.StructField("grau",                 T.StringType()),
    T.StructField("classe",               T.StructType([
        T.StructField("codigo", T.IntegerType()),
        T.StructField("nome",   T.StringType())])),
    T.StructField("sistema",              T.StructType([
        T.StructField("codigo", T.IntegerType()),
        T.StructField("nome",   T.StringType())])),
    T.StructField("formato",              T.StructType([
        T.StructField("codigo", T.IntegerType()),
        T.StructField("nome",   T.StringType())])),
    T.StructField("orgaoJulgador",        T.StructType([
        T.StructField("codigo",              T.IntegerType()),
        T.StructField("nome",                T.StringType()),
        T.StructField("codigoMunicipioIBGE", T.LongType())])),
    T.StructField("assuntos",             T.ArrayType(T.StructType([
        T.StructField("codigo", T.IntegerType()),
        T.StructField("nome",   T.StringType())]))),
    T.StructField("movimentos",           T.ArrayType(schema_mov)),
])

bronze = spark.table("bronze_datajud_raw").withColumn(
    "p", F.from_json("raw_payload", schema_proc))

print("Registros no bronze:", bronze.count())

Registros no bronze: 243831


### BLOCO 2 — Dedupe: última captura de cada processo

O bronze contém lotes ingestados: **243.831 registros × 225.556 processos
distintos** (18.275 duplicatas de reingestão).

**Regra de dedupe:** `row_number` particionado por `numeroProcesso`,
ordenado por `ingestion_timestamp` descendente — vence a captura mais
recente de cada processo (a versão mais atual do Datajud).

**QA herdado:** o print de `qtd_distintos` valida contra o alvo 225.556.

In [0]:
# ==============================================================================
# BLOCO 2 — DEDUPE: última captura de cada processo
# Bronze: 243.831 registros x 225.556 processos distintos (lotes reingestados).
# Regra: row_number por processo, ordenando pela captura mais recente.
# ==============================================================================

w_ultima_captura = Window.partitionBy("p.numeroProcesso") \
                         .orderBy(F.col("ingestion_timestamp").desc())

proc_dedupe = (bronze
    .withColumn("rn", F.row_number().over(w_ultima_captura))
    .filter("rn = 1")
    .drop("rn"))

qtd_distintos = proc_dedupe.count()
print("Processos distintos após dedupe (alvo 225.556):", qtd_distintos)

Processos distintos após dedupe (alvo 225.556): 225556


### BLOCO 3 — Silver de processos (`silver_datajud_processos`)

Converte o processo deduplicado em 1 linha tabular, com três decisões de
qualidade:

1. **Parser multi-formato de data** (COALESCE de tentativas): o Datajud
   retorna datas em formatos heterogêneos — ISO com/sem hora, 14 dígitos
   (`yyyyMMddHHmmss`) e 8 dígitos (`yyyyMMdd`). `try_to_date` retorna NULL
   quando o formato não casa e o COALESCE devolve o primeiro parse
   bem-sucedido; a ordem resolve a precedência (14 dígitos antes de 8).
2. **Escopo temporal da pesquisa:** `escopo_lei_atual` marca processos
   ajuizados a partir de **09/06/2005** (vigência da Lei 11.101/2005);
   `regime_legal` distingue da Lei 7.661/1945 (legado).
3. **Gravação com overwriteSchema:** o schema evolui com o notebook sem
   quebrar re-execuções.

**QA do parser:** contagem de `data_ajuizamento` NULL — falhas de parse
devem ficar em ~0; qualquer valor relevante é defeito de origem a documentar.

In [0]:
# ==============================================================================
# CORREÇÃO BLOCO 3 — parser multi-formato via COALESCE
# try_to_date retorna NULL quando o formato não casa; coalesce retorna o
# primeiro parse bem-sucedido. Sem when/otherwise — cada tentativa é
# independente e a ordem resolve a precedência (14 dígitos antes de 8).
# ==============================================================================

DATA_AJUIZ = "p.dataAjuizamento"

parse_data = F.coalesce(
    # ISO com hora: '1998-12-01T00:00:00' ou '1998-12-01'
    F.try_to_date(F.regexp_extract(DATA_AJUIZ, r"^(\d{4}-\d{2}-\d{2})", 1),
                  "yyyy-MM-dd"),
    # Sequência de 14 dígitos: '19981201000000' (yyyyMMddHHmmss)
    F.try_to_date(F.regexp_extract(DATA_AJUIZ, r"^(\d{14})", 1),
                  "yyyyMMddHHmmss"),
    # Sequência de 8 dígitos: '19981201' (yyyyMMdd)
    F.try_to_date(F.regexp_extract(DATA_AJUIZ, r"^(\d{8})", 1),
                  "yyyyMMdd"),
    # ISO puro como fallback final
    F.try_to_date(DATA_AJUIZ, "yyyy-MM-dd"))

silver_proc = (proc_dedupe.select(
        F.col("p.numeroProcesso").alias("numero_processo"),
        F.col("tribunal_origem"),
        F.col("p.grau").alias("grau_jurisdicao"),
        F.col("p.classe.codigo").alias("classe_codigo"),
        F.col("p.classe.nome").alias("classe_nome"),
        F.col("p.sistema.nome").alias("sistema_processual"),
        F.col("p.formato.nome").alias("formato_processo"),
        parse_data.alias("data_ajuizamento"),
        F.col("p.orgaoJulgador.codigo").alias("orgao_julgador_codigo"),
        F.col("p.orgaoJulgador.nome").alias("orgao_julgador_nome"),
        F.col("p.orgaoJulgador.codigoMunicipioIBGE").alias("municipio_ibge_codigo"),
        F.col("p.assuntos").alias("assuntos"),
        F.col("p.movimentos").alias("movimentos"))
    .withColumn("escopo_lei_atual",
        F.col("data_ajuizamento") >= F.lit("2005-06-09").cast("date"))
    .withColumn("regime_legal",
        F.when(F.col("escopo_lei_atual"), F.lit("Lei 11.101/2005"))
         .otherwise(F.lit("Lei 7.661/1945 (legado)"))))

# QA do parser: NULLs devem ficar próximos de 0
print("Total processos:", silver_proc.count())
print("data_ajuizamento NULL (falha de parse):",
      silver_proc.filter("data_ajuizamento IS NULL").count())

(silver_proc.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver_datajud_processos"))

Total processos: 225556
data_ajuizamento NULL (falha de parse): 0


### BLOCO 4 — Fato de movimentos (`silver_fato_movimento`)

Explode o array de movimentos para o grão de análise principal:
**1 linha por movimento** (~14–17 milhões de eventos).

**Higiene herdada do parse:**
- `precisao_data`: dia_hora / dia / mês / ano / inválida, conforme o
  comprimento da string original
- `precisao_suficiente`: só dia e dia_hora suportam marcos datados —
  flag usada depois no Bloco 7
- `sem_codigo`: movimento sem código TPU (rastro de qualidade de origem)
- `complementos_txt`: complementos tabelados concatenados em texto
  pesquisável (insumo das regras textuais do Bloco 6)

**Dedupe por conteúdo:** movimentos idênticos repetidos no mesmo processo
(eco das capturas duplicadas da Bronze) colapsam em 1 linha, e a contagem
vira `qtd_ocorrencias` — nada é descartado, a repetição fica registrada
como evidência de QA.

In [0]:
# ==============================================================================
# BLOCO 4 — SILVER_FATO_MOVIMENTO (1 linha por movimento)
# Explode dos movimentos com QA estrutural herdada: precisão da data,
# flag de movimento sem código TPU e contagem de ocorrências repetidas
# (rastro das capturas duplicadas da bronze, consolidadas na dedupe).
# ==============================================================================

silver_mov = (
    silver_proc.drop("assuntos", "movimentos")
    .join(silver_proc.select("numero_processo",
                             F.explode_outer("movimentos").alias("m")),
          "numero_processo")
    .select("numero_processo", "tribunal_origem",
            F.col("classe_codigo").alias("classe_codigo_mov"),
            F.col("m.codigo").alias("movimento_codigo"),
            F.col("m.nome").alias("movimento_nome"),
            F.col("m.dataHora").alias("data_hora_raw"),
            F.col("m.complementosTabelados").alias("complementos"))
    .withColumn("precisao_data",
        F.when(F.length("data_hora_raw") > 10, F.lit("dia_hora"))
         .when(F.length("data_hora_raw") == 10, F.lit("dia"))
         .when(F.length("data_hora_raw") == 7,  F.lit("mes"))
         .when(F.length("data_hora_raw") == 4,  F.lit("ano"))
         .otherwise(F.lit("invalida")))
    .withColumn("data_movimento",
        F.when(F.col("precisao_data").isin("dia", "dia_hora"),
               F.to_date(F.substring("data_hora_raw", 1, 10))))
    .withColumn("precisao_suficiente",
        F.col("precisao_data").isin("dia", "dia_hora"))
    .withColumn("sem_codigo", F.col("movimento_codigo").isNull())
    .withColumn("complementos_txt",
        F.concat_ws(" | ", F.transform("complementos",
            lambda c: F.concat_ws(": ",
                F.coalesce(c["nome"],      F.lit("")),
                F.coalesce(c["descricao"], F.lit(""))))))
    .drop("complementos", "data_hora_raw"))

# dedupe de movimento idêntico no mesmo processo; repetições viram QA
mov_cols = [c for c in silver_mov.columns]
(silver_mov
    .groupBy(*mov_cols)
    .agg(F.count(F.lit(1)).alias("qtd_ocorrencias"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver_fato_movimento"))

eventos = spark.table("silver_fato_movimento").count()    

print(f"silver_fato_movimento gravada. Alvo de referência: ~{eventos/1000000:.1f} milhões de eventos.")

silver_fato_movimento gravada. Alvo de referência: ~14.1 milhões de eventos.


### BLOCO 5 — Dimensões derivadas da Silver

Quatro tabelas de apoio, todas **derivadas** (sem carga externa):

- **`silver_dim_orgao_julgador`**: 1 linha por órgão × tribunal, com
  contagem de processos. Agrupar por (código, nome, tribunal, município)
   — nomes variantes do
  mesmo órgão não multiplicam mais as contagens.
- **`silver_dim_assunto`**: catálogo de assuntos com nº de processos
  distintos (`countDistinct` evita dupla contagem em processos multi-assunto).
- **`silver_ponte_processo_assunto`**: ponte N:N processo–assunto, base
  para definir o universo da pesquisa por matéria.
- **`silver_dim_processo`**: grade enxuta do processo (tribunal, grau,
  sistema, formato) — leitura rápida sem carregar os arrays.

As dimensões alimentam o star schema da Gold (003) sem recálculo lá.

In [0]:
# ==============================================================================
# BLOCO 5 — DIMS DERIVADAS DA SILVER
# Órgãos julgadores, assuntos, ponte processo-assunto e grade de processo.
# ==============================================================================

(silver_proc.groupBy("orgao_julgador_codigo", "orgao_julgador_nome",
                     "tribunal_origem", "municipio_ibge_codigo")
    .agg(F.count("*").alias("qtd_processos"))
    .select(F.col("orgao_julgador_codigo"), F.col("orgao_julgador_nome"),
            F.col("tribunal_origem").alias("tribunal_sigla"),
            F.col("municipio_ibge_codigo"), F.col("qtd_processos"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver_dim_orgao_julgador"))

(silver_proc.select("numero_processo",
                    F.explode_outer("assuntos").alias("a"))
    .groupBy(F.col("a.codigo").alias("assunto_codigo"),
             F.col("a.nome").alias("assunto_nome"))
    .agg(F.countDistinct("numero_processo").alias("qtd_processos"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver_dim_assunto"))

(silver_proc.select("numero_processo",
                    F.explode_outer("assuntos").alias("a"))
    .select("numero_processo",
            F.col("a.codigo").alias("assunto_codigo"),
            F.col("a.nome").alias("assunto_nome"))
    .distinct()
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver_ponte_processo_assunto"))

(silver_proc.select("numero_processo", "tribunal_origem", "grau_jurisdicao",
                    "sistema_processual", "formato_processo")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver_dim_processo"))

print("Dims da silver gravadas: orgao, assunto, ponte, processo.")

Dims da silver gravadas: orgao, assunto, ponte, processo.


### BLOCO 6 — Regras de marcação (`silver_regras_marcacao`, v2)

Tabela de configuração que transforma conhecimento jurídico em regras
declaradas: `regra_id`, `categoria_evento`, `descricao` e `condicao_sql`.
O pipeline fica auditorável — toda marcação rastreia a regra que a gerou.

**Por que uma tabela de regras e não lógica embutida no código:** as
regras de marcação são a ponte entre o direito (Lei 11.101/2005) e o dado.
Isolá-las permite revisar e discutir uma regra de cada vez sem tocar no
pipeline, e todo evento classificado carrega `regra_aplicada` — qualquer
métrica da Gold é rastreável até o critério que a originou.

**Princípio de precedência:** o Bloco 7 aplica as regras **na ordem da
lista** — a primeira que casa vence. É por isso que a R12 (homologação)
está em **primeiro lugar**: sem isso, a R10 (TPU 12444, deferimento
genérico) "engoliria" o evento de homologação antes de a regra
específica enxergá-lo. Regras específicas sempre precedem as genéricas.

### As regras (na ordem de aplicação)

**R12 — Homologação do plano de recuperação (classe 128/129).**

*O que marca:* a aprovação judicial do plano, marco do art. 57 da Lei 11.101/2005 — encerra a fase de "stay period" e inicia o cumprimento.

*Como funciona:* exige `classe_codigo_mov IN (128 , 129)` (recuperação extrajudicial/recuperação judicial) e casa por padrões textuais em `movimento_nome` **e** `complementos_txt`
("homolog+plano", "plano+homologado") — o Datajud não tem código TPU dedicado a esse ato, então a identificação é necessariamente textual.

*Por que existe:* é um dos desfechos centrais da pesquisa,contudo diversas tentativas mostraram que esse movimento não ocorre nos dados (0 ocorrências); a regra segue ativa — se o CNJ passar a registrar o ato, o pipeline já o captura.

**R11 — Decretação de falência (TPU 202).**

*O que marca:* a sentença declaratória de falência (art. 99 da LRE) — conversão da recuperação em falência ou falência direta.

*Como funciona:* código TPU exato (`movimento_codigo = 202`) — não precisa de padrão textual, o vocabulário processual unificado tem código dedicado a esse ato.

*Por que existe:* é o desfecho adverso do modelo. Permite medir **tempo até a falência** e quantificar quantas recuperações processadas não se concretizam.

**R13 — Extinção do processo (TPU 22/456).**

*O que marca:* o fim do processo no juízo.

*Como funciona:* `movimento_codigo IN (22, 456)` — dois códigos TPU para baixa definitiva e extinção respectivamente, elas cobrem as variantes de extinção, uma vez que o CNJ não uniformiza o ato em um só código.

*Por que existe:* delimita o **fim do ciclo de vida** do processo. Sem ela, durações mediriam até a última captura, não até o desfecho real.

**R14 — Encerramento/arquivamento dos autos.**

*O que marca:* o ato administrativo de encerramento ou arquivamento.

*Como funciona:* padrão textual ("%encerra%", "%arquivamento%") — o arquivamento não tem código TPU único.

*Por que existe:* distingue o arquivamento formal sem decisão de mérito (abandono da ação, falta de interesse, ausência de pressupostos processuais, desistênci), da extinção após a decisão de mérito (procedente/improcedente).

*Obs*: processos podem ser extintos e encerrados em momentos distintos; confundi-los distorceria as durações.

**R9 — Processamento da recuperação judicial (classe 129).**

*O que marca:* o deferimento do processamento (art. 52 da LRE) — o juiz acolhe o pedido e instaura a recuperação.

*Como funciona:* exige `classe_codigo_mov = 129` (processo de RJ) **e** padrão textual "%processamento%" — restringe por classe para não pegar "processamentos" de outras matérias.

*Por que existe:* define o **ponto zero** de todas as durações da Gold: tempo de tramitação, tempo até homologação, tempo até o desfecho. Está depois de R12/R11/R13/R14 na lista porque, se o processo já tem desfecho registrado, o desfecho vence o processamento na marcação.

**R10 — Deferimento genérico (TPU 12444).**

*O que marca:* qualquer decisão de deferimento que **não** foi capturada pela R12 (homologação) nem pela R9 (processamento).

*Como funciona:* código TPU exato (`movimento_codigo = 12444`), que tem o nome de deferimeto nessa tabela.

*Por que existe:* é a **regra-guarda**. Deferimentos genéricos são frequentes no Datajud; sem capturá-los, eventos relevantes ficariam sem categoria. A precedência controlada (por último na família dos deferimentos) impede que ele absorva homologação ou processamento.

**R15 — Tutela/liminar de urgência registrada nos autos.**

*O que marca:* a **protocolização** do pedido de tutela de urgência — o ato de registro, sem decisão do juiz.

*Como funciona:* padrão textual ("%tutela%" OR "%liminar%") em
`movimento_nome`.

*Por que existe:* abre a trilha da tutela e é a **âncora do início** da tentativa de proteção. Na recuperação judicial, a tutela de urgência concedida inicia o *stay period* — a suspensão das ações e execuções contra o devedor (art. 6º, § 4º da LRE). Sem a marca de registro, não se mede quanto tempo o pedido ficou pendente.

**R16 — Tutela deferida/concedida.**

*O que marca:* a decisão favorável — a tutela é concedida e a proteção efetivamente começa.

*Como funciona:* padrão textual ("%conced%" em `movimento_nome`,
"%concedida%" em `complementos_txt`) — cobre "concede", "concedida" e variantes da mesma raiz.

*Por que existe:* define o **início efetivo do stay period**. A Gold mede a duração da proteção a partir daqui — deferimento é o único evento que transforma o pedido em proteção real.

**R17 — Tutela indeferida.**

*O que marca:* o desfecho adverso — o juiz nega a tutela.

*Como funciona:* padrão textual ("%indefere%", "%não conced%").

*Por que existe:* o indeferimento define o fim da proteção pontual (quando havia deferimento antes) ou a sua ausência naquele conflito específico. Separar deferimento de indeferimento permite quantificar em quantos processos a empresa em crise financeira perdeu a proteção do instituto da recuperação.


In [0]:
# ==============================================================================
# BLOCO 6 — REGRAS DE MARCACAO (v2 corrigida)
# FIX R12: regra de homologação movida para PRIMEIRA posição (precedência
# sobre R10/TPU 12444, que engolia o evento) e condição alargada:
# '%homolog%plano%', '%plano%homologado%' e complementos textualizados.
# IMPORTANTE: o loop de classificação aplica as regras NA ORDEM da lista —
# regras específicas antes das genéricas.
# ==============================================================================

regras = [
    # --- ESPECÍFICAS primeiro (desfechos que usam movimentos genéricos) ---
    ("R12", "homologacao_plano",
     "Homologação do plano de recuperação (RJ ou RE)",
     "classe_codigo_mov IN (128, 129) AND (lower(movimento_nome) LIKE '%homolog%plano%' "
     "OR lower(movimento_nome) LIKE '%plano%homologado%' "
     "OR lower(movimento_nome) LIKE '%homologacao de plano%' "
     "OR lower(complementos_txt) LIKE '%homolog%plano%' "
     "OR lower(complementos_txt) LIKE '%plano%homologado%')"),
    ("R11", "decretao_falencia",
     "TPU 202 - Decretação de falência",
     "movimento_codigo = 202"),
    ("R13", "extincao",
     "Extinção do processo (TPU 22/456)",
     "movimento_codigo IN (22, 456)"),
    ("R14", "encerramento",
     "Encerramento/arquivamento dos autos",
     "lower(movimento_nome) LIKE '%encerra%' OR lower(movimento_nome) LIKE '%arquivamento%'"),
    # --- DEFERIMENTO de processamento: textual antes do TPU genérico ---
    ("R9",  "processamento_rj",
     "Despacho textual de deferimento do processamento de RJ (classe 129)",
     "classe_codigo_mov = 129 AND lower(movimento_nome) LIKE '%processamento%'"),
    ("R10", "deferimento_estrito",
     "TPU 12444 - Decisão de deferimento (exceto homologação, capturada antes pela R12)",
     "movimento_codigo = 12444"),
    # --- TUTELAS ---
    ("R15", "tutela_urgencia",
     "Tutela/liminar de urgência registrada nos autos",
     "lower(movimento_nome) LIKE '%tutela%' OR lower(movimento_nome) LIKE '%liminar%'"),
    ("R16", "tutela_deferida",
     "Deferimento/concessão de tutela",
     "lower(movimento_nome) LIKE '%conced%' OR lower(complementos_txt) LIKE '%concedida%'"),
    ("R17", "tutela_indeferida",
     "Indeferimento de tutela",
     "lower(movimento_nome) LIKE '%indefere%' OR lower(movimento_nome) LIKE '%não conced%'"),
]

schema_regra = T.StructType([
    T.StructField("regra_id",         T.StringType()),
    T.StructField("categoria_evento", T.StringType()),
    T.StructField("descricao",        T.StringType()),
    T.StructField("condicao_sql",     T.StringType()),
])

spark.createDataFrame(regras, schema_regra) \
     .write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable("silver_regras_marcacao")

print("silver_regras_marcacao regravada:", len(regras),
      "regras (R12 em precedência máxima).")

silver_regras_marcacao regravada: 9 regras (R12 em precedência máxima).


### BLOCO 7 — Classificação de eventos e marcos processuais

Duas saídas em cascata:

1. **`silver_datajud_eventos_enriquecidos`**: a fato de movimentos com as
   colunas `categoria_evento` e `regra_aplicada` preenchidas. A primeira
   regra que casa vence — implementado com `when(...).otherwise(...)` em
   cadeia, na ordem da lista do Bloco 6.
2. **`silver_marcos_processo`**: os eventos **classificados e datados**
   (filtro `precisao_suficiente`), com DDL + INSERT separados (padrão
   RTAS-safe já validado no 001) e COMMENT em todas as colunas.

**Por que marcos:** a Gold calcula durações (tempo até processamento,
stay period, tempo até falência) a partir dessas datas — aqui é o único
ponto do pipeline onde evento jurídico vira data mensurável.

**Verificação imediata:** o `display` de marcos por regra permite conferir
de olho se a R12 materializou ou se segue como não-achado (0 linhas).

In [0]:
# ==============================================================================
# BLOCO 7 — CLASSIFICACAO DOS EVENTOS + MARCOS PROCESSUAIS
# Regenera os eventos enriquecidos e os marcos com as regras corrigidas.
# A primeira regra que casa vence, na ordem da lista (específicas primeiro).
# ==============================================================================

df_evt = (spark.table("silver_fato_movimento")
    .withColumn("categoria_evento", F.lit(None).cast("string"))
    .withColumn("regra_aplicada",   F.lit(None).cast("string")))

for regra_id, categoria, _, condicao in regras:
    df_evt = (df_evt
        .withColumn("regra_aplicada",
            F.when(F.expr(condicao), F.lit(regra_id))
             .otherwise(F.col("regra_aplicada")))
        .withColumn("categoria_evento",
            F.when(F.expr(condicao) & F.col("categoria_evento").isNull(),
                   F.lit(categoria))
             .otherwise(F.col("categoria_evento"))))

(df_evt.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver_datajud_eventos_enriquecidos"))

# DDL + carga separados (padrão RTAS-safe já validado)
spark.sql("""
  CREATE OR REPLACE TABLE silver_marcos_processo (
    numero_processo    STRING COMMENT 'Número CNJ do processo',
    classe_codigo_mov  BIGINT COMMENT 'Classe processual do processo',
    categoria_evento   STRING COMMENT 'Categoria jurisimétrica da regra aplicada',
    regra_aplicada     STRING COMMENT 'regra_id (linhagem/auditoria)',
    data_movimento     DATE   COMMENT 'Data normalizada do marco',
    movimento_codigo   BIGINT COMMENT 'Código TPU do movimento',
    movimento_nome     STRING COMMENT 'Texto do movimento',
    complementos_txt   STRING COMMENT 'Complementos tabelados concatenados'
  ) USING DELTA
  COMMENT 'Marcos processuais datados: 1 linha por evento classificado'
""")

spark.sql("""
  INSERT OVERWRITE silver_marcos_processo
  SELECT numero_processo, classe_codigo_mov, categoria_evento,
         regra_aplicada, data_movimento, movimento_codigo,
         movimento_nome, complementos_txt
  FROM silver_datajud_eventos_enriquecidos
  WHERE categoria_evento IS NOT NULL AND precisao_suficiente
""")

# Verificação imediata da correção: homologação deve materializar agora
print("Marcos por regra:")
display(spark.sql("""
  SELECT categoria_evento, regra_aplicada, count(*) AS n
  FROM silver_marcos_processo GROUP BY 1, 2 ORDER BY n DESC
"""))

Marcos por regra:


categoria_evento,regra_aplicada,n
extincao,R13,83891
tutela_urgencia,R15,9932
encerramento,R14,8505
deferimento_estrito,R10,7884
decretao_falencia,R11,5061


### BLOCO 8 — QA da camada Silver (`silver_qa_execucao`)

Gate de qualidade antes de qualquer consumo pela Gold. Cinco métricas com
valores esperados quando existem:

| Métrica | Esperado | Interpretação |
|---|---|---|
| processos_distintos | 225.556 | dedupe correto (igual à Bronze distinta) |
| movimentos_totais | ~14,1 mi (referência) | grão preservado da origem |
| movimentos_com_data_valida | — | sanidade do parser de datas |
| eventos_classificados_por_regra | — | cobertura das regras de marcação |
| marcos_processuais_datados | — | insumo das durações na Gold |

Cada execução grava `executado_em` — o histórico de QA é comparável entre
execuções.



In [0]:
# ==============================================================================
# BLOCO 8 — QA DA CAMADA SILVER (gravada: silver_qa_execucao)
# Gate de qualidade antes da gold: contagens-chave com valores esperados.
# ==============================================================================

qa = spark.sql("""
  SELECT 'processos_distintos'      AS metrica,
         count(*) AS valor, 225556 AS esperado
  FROM silver_datajud_processos
  UNION ALL
  SELECT 'movimentos_totais',
         (SELECT count(*) FROM silver_fato_movimento),
         14129958
  UNION ALL
  SELECT 'movimentos_com_data_valida',
         (SELECT count(*) FROM silver_fato_movimento WHERE precisao_suficiente),
         NULL
  UNION ALL
  SELECT 'eventos_classificados_por_regra',
         (SELECT count(*) FROM silver_datajud_eventos_enriquecidos
          WHERE categoria_evento IS NOT NULL),
         NULL
  UNION ALL
  SELECT 'marcos_processuais_datados',
         (SELECT count(*) FROM silver_marcos_processo),
         NULL
""")

(qa.withColumn("executado_em", F.current_timestamp())
   .write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("silver_qa_execucao"))

display(spark.table("silver_qa_execucao"))
print("02 concluído: silver 100% reconstruída a partir de bronze_datajud_raw.")

metrica,valor,esperado,executado_em
eventos_classificados_por_regra,115273,null,2026-09-26T20:22:45.406Z
movimentos_com_data_valida,14128738,null,2026-09-26T20:22:45.406Z
marcos_processuais_datados,115273,null,2026-09-26T20:22:45.406Z
processos_distintos,225556,225556,2026-09-26T20:22:45.406Z
movimentos_totais,14129958,14129958,2026-09-26T20:22:45.406Z


02 concluído: silver 100% reconstruída a partir de bronze_datajud_raw.


### 9. Bloco de governança — comentários das colunas das tabelas Silver

In [0]:
# BLOCO DE GOVERNANÇA — comentários de coluna da Silver (idempotente)
# Executar SEMPRE ao final do notebook 02: escrita por DataFrame não
# propaga comentários, e overwriteSchema os apaga a cada re-execução.

tabelas = {
  "silver_datajud_processos": {
    "numero_processo":       "Número CNJ de 20 dígitos — identificador único do processo",
    "tribunal_origem":       "Sigla do Tribunal de Justiça de onde o registro foi extraído (ex: TJSP, TJRJ, TJDFT)",
    "grau_jurisdicao":       "Grau de jurisdição (G1 = primeiro grau, G2 = segundo grau)",
    "classe_codigo":         "Código TPU da classe processual (129 = RJ, 108 = Falência, 128 = RE)",
    "classe_nome":           "Nome da classe processual conforme a TPU do CNJ",
    "sistema_processual":    "Sistema processual em que o processo tramita (ex: PJe, e-SAJ)",
    "formato_processo":      "Formato do processo (eletrônico ou físico)",
    "data_ajuizamento":      "Data de ajuizamento; dela deriva o recorte escopo_lei_atual",
    "orgao_julgador_codigo": "Código CNJ do órgão julgador (vara/juízo)",
    "orgao_julgador_nome":   "Nome do órgão julgador; dele deriva a especialização (eh_especializada na Gold)",
    "municipio_ibge_codigo": "Código IBGE do município da sede do órgão julgador",
    "assuntos":              "Array de assuntos do processo (código + nome); o recorte da coleta filtra pelos códigos 4992–4997",
    "movimentos":            "Array do histórico completo de movimentações — insumo bruto das regras de marcação",
    "escopo_lei_atual":      "TRUE se ajuizado a partir de 09/06/2005 (vigência da Lei 11.101/2005) — universo das métricas",
    "regime_legal":          "Regime legal aplicável (Lei 11.101/2005 ou Lei 7.661/1945 legado)",
  },
  "silver_fato_movimento": {
    "numero_processo":      "Número CNJ do processo ao qual o movimento pertence",
    "tribunal_origem":      "Sigla do tribunal do processo",
    "classe_codigo_mov":    "Código TPU da classe do processo no momento do movimento",
    "movimento_codigo":     "Código TPU do movimento",
    "movimento_nome":       "Descrição textual do ato processual",
    "data_movimento":       "Data do movimento após o parser multi-formato",
    "precisao_data":        "Precisão da data original (dia_hora / dia / mes / ano / invalida)",
    "precisao_suficiente":  "TRUE se a precisão da data basta para construir marco datado",
    "sem_codigo":           "TRUE se o movimento não possui código TPU (rastro de qualidade da origem)",
    "complementos_txt":     "Complementos tabelados concatenados em texto pesquisável",
    "qtd_ocorrencias":      "Ocorrências idênticas colapsadas pelo dedupe por conteúdo (evidência, não perda)",
  },
  "silver_dim_orgao_julgador": {
    "orgao_julgador_codigo": "Código CNJ do órgão julgador",
    "orgao_julgador_nome":   "Nome do órgão julgador conforme a origem",
    "tribunal_sigla":        "Sigla do tribunal de origem",
    "municipio_ibge_codigo": "Código IBGE do município da sede",
    "qtd_processos":         "Quantidade de processos vinculados ao órgão",
  },
  "silver_dim_assunto": {
    "assunto_codigo": "Código do assunto no padrão CNJ",
    "assunto_nome":   "Nome do assunto",
    "qtd_processos":  "Processos distintos com o assunto (countDistinct evita dupla contagem)",
  },
  "silver_ponte_processo_assunto": {
    "numero_processo": "Número CNJ do processo",
    "assunto_codigo":  "Código do assunto vinculado ao processo",
    "assunto_nome":    "Nome do assunto vinculado",
  },
  "silver_dim_processo": {
    "numero_processo":    "Número CNJ do processo",
    "tribunal_origem":    "Sigla do tribunal de origem",
    "grau_jurisdicao":    "Grau de jurisdição (G1/G2)",
    "sistema_processual": "Sistema processual em que o processo tramita",
    "formato_processo":   "Formato do processo (eletrônico ou físico)",
  },
  "silver_regras_marcacao": {
    "regra_id":         "Identificador da regra (R9–R17)",
    "categoria_evento": "Categoria de evento atribuída pela regra",
    "descricao":        "Descrição humana da regra",
    "condicao_sql":     "Condição SQL declarada — aplicada na ordem da lista; a primeira que casa vence",
  },
  "silver_datajud_eventos_enriquecidos": {
    "numero_processo":     "Número CNJ do processo ao qual o movimento pertence",
    "tribunal_origem":     "Sigla do tribunal do processo",
    "classe_codigo_mov":   "Código TPU da classe do processo",
    "movimento_codigo":    "Código TPU do movimento",
    "movimento_nome":      "Descrição textual do ato processual",
    "data_movimento":      "Data do movimento após o parser multi-formato",
    "precisao_data":       "Precisão da data original (dia_hora / dia / mes / ano / invalida)",
    "precisao_suficiente": "TRUE se a precisão da data basta para construir marco datado",
    "sem_codigo":          "TRUE se o movimento não possui código TPU",
    "complementos_txt":    "Complementos tabelados concatenados em texto pesquisável",
    "qtd_ocorrencias":     "Ocorrências idênticas colapsadas pelo dedupe por conteúdo",
    "categoria_evento":    "Categoria do evento classificado — primeira regra que casa vence",
    "regra_aplicada":      "regra_id da regra que classificou o evento (linhagem/auditoria)",
  },
  "silver_marcos_processo": {  # já nasce comentada pelo DDL; reaplicação idempotente
    "numero_processo":   "Número CNJ do processo",
    "classe_codigo_mov": "Classe processual do processo",
    "categoria_evento":  "Categoria jurimétrica da regra aplicada",
    "regra_aplicada":    "regra_id (linhagem/auditoria)",
    "data_movimento":    "Data normalizada do marco",
    "movimento_codigo":  "Código TPU do movimento",
    "movimento_nome":    "Texto do movimento",
    "complementos_txt":  "Complementos tabelados concatenados",
  },
  "silver_qa_execucao": {
    "metrica":      "Nome da métrica de QA da camada",
    "valor":        "Valor obtido na execução",
    "esperado":     "Valor esperado (quando definido)",
    "executado_em": "Timestamp da execução — histórico comparável entre execuções",
  },
}

falhas = []
for tabela, cols in tabelas.items():
    for coluna, comment in cols.items():
        try:
            spark.sql(f"ALTER TABLE default.{tabela} ALTER COLUMN {coluna} COMMENT '{comment}'")
        except Exception as e:
            falhas.append((tabela, coluna, str(e).splitlines()[0]))

total = sum(len(v) for v in tabelas.values())
if falhas:
    print(f"{len(falhas)} de {total} comentários não aplicados — conferir nomes de coluna:")
    for t, c, msg in falhas:
        print(f"  {t}.{c}: {msg}")
else:
    print(f"Governança aplicada: {total} comentários de coluna em {len(tabelas)} tabelas da Silver.")

Governança aplicada: 71 comentários de coluna em 10 tabelas da Silver.
